# 02 — Custom CNN From Scratch

Builds and trains a CNN (no pretrained weights) with a softmax head for 8-class fundus classification,
using class-weighted loss to address imbalance. Saves the model + metrics so `04_model_comparison.ipynb`
can compare it against the fine-tuned transfer-learning model.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

CONFIG = {
    "DATA_DIR": "/kaggle/working/dataset_extracted/splits",
    "IMG_DIR": "/kaggle/working/dataset_extracted/images",
    "LABEL_COL": "unified_label",
    "FILENAME_COL": "filename",
    "IMG_SIZE": (224, 224),
    "BATCH_SIZE": 32,
    "EPOCHS": 30,
    "SEED": 42,
}

CLASSES = ["Normal", "Diabetic Retinopathy", "Others", "Glaucoma",
           "Cataract", "Myopia", "AMD", "Hypertension"]

os.makedirs("artifacts", exist_ok=True)
tf.random.set_seed(CONFIG["SEED"])

In [ ]:
train_df = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "train.csv"))
val_df   = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "val.csv"))
test_df  = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "test.csv"))

for df in (train_df, val_df, test_df):
    df[CONFIG["LABEL_COL"]] = df[CONFIG["LABEL_COL"]].astype(str)

print(train_df[CONFIG["LABEL_COL"]].value_counts())

## Data generators (augmentation for the scratch CNN, which has no pretrained priors)

In [ ]:
train_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=(0.85, 1.15),
)
eval_aug = ImageDataGenerator(rescale=1./255)

def make_gen(datagen, df, shuffle):
    return datagen.flow_from_dataframe(
        df,
        directory=CONFIG["IMG_DIR"],
        x_col=CONFIG["FILENAME_COL"],
        y_col=CONFIG["LABEL_COL"],
        target_size=CONFIG["IMG_SIZE"],
        batch_size=CONFIG["BATCH_SIZE"],
        class_mode="categorical",
        classes=CLASSES,
        shuffle=shuffle,
        seed=CONFIG["SEED"],
    )

train_gen = make_gen(train_aug, train_df, shuffle=True)
val_gen   = make_gen(eval_aug, val_df, shuffle=False)
test_gen  = make_gen(eval_aug, test_df, shuffle=False)

## Class weights (handling imbalance — required by the brief)

In [ ]:
y_train_idx = train_gen.classes
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_idx),
    y=y_train_idx,
)
class_weight_dict = dict(zip(np.unique(y_train_idx), weights))
print({CLASSES[k]: round(v, 2) for k, v in class_weight_dict.items()})

## Custom CNN architecture

In [ ]:
def build_scratch_cnn(input_shape=(224, 224, 3), n_classes=8):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)

    return models.Model(inputs, outputs, name="scratch_cnn")

scratch_model = build_scratch_cnn(input_shape=CONFIG["IMG_SIZE"] + (3,), n_classes=len(CLASSES))
scratch_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
scratch_model.summary()

## Train

In [ ]:
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    callbacks.ModelCheckpoint("artifacts/scratch_cnn_best.keras", monitor="val_loss", save_best_only=True),
]

history = scratch_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=CONFIG["EPOCHS"],
    class_weight=class_weight_dict,
    callbacks=cbs,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.savefig("artifacts/scratch_cnn_training_curves.png", dpi=150)
plt.show()

## Evaluate — per-class precision / recall / F1 (required deliverable)

In [ ]:
test_gen.reset()
y_true = test_gen.classes
y_prob = scratch_model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
print(classification_report(y_true, y_pred, target_names=CLASSES, zero_division=0))
report_df.to_csv("artifacts/scratch_cnn_classification_report.csv")
report_df

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Scratch CNN — Confusion Matrix (Test Set)")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig("artifacts/scratch_cnn_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Inference speed benchmark (for the real-time deployment discussion in 04)
n_bench = 50
sample_batch = next(iter(test_gen))[0][:min(n_bench, CONFIG["BATCH_SIZE"])]
scratch_model.predict(sample_batch, verbose=0)  # warmup

start = time.time()
for _ in range(10):
    scratch_model.predict(sample_batch, verbose=0)
elapsed = (time.time() - start) / 10
ms_per_image = elapsed / len(sample_batch) * 1000
print(f"Scratch CNN: {ms_per_image:.2f} ms/image, {1000/ms_per_image:.1f} FPS")

metrics_summary = {
    "model": "scratch_cnn",
    "test_accuracy": float(report["accuracy"]),
    "macro_f1": float(report["macro avg"]["f1-score"]),
    "weighted_f1": float(report["weighted avg"]["f1-score"]),
    "ms_per_image": ms_per_image,
    "n_params": int(scratch_model.count_params()),
}
with open("artifacts/scratch_cnn_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
metrics_summary

Model, curves, report, confusion matrix and timing are all saved under `artifacts/`. Next: `03_transfer_learning.ipynb`.